# RSNA Pneumonia Detection — YOLO Pipeline (Colab)

End-to-end пайплайн: скачивание датасета → фильтрация разметки → обучение YOLO → оценка.

**Репозиторий:** https://github.com/e-eki/rsna-pneumonia-yolo-pipeline (ветка `development`)

**Runtime:** `Runtime → Change runtime type → T4 GPU`.

**Файл `.ipynb` — только для Colab.** Kaggle-версия будет в отдельном ноутбуке (`kaggle_train.ipynb`).

## 0. Проверка GPU

In [ ]:
!nvidia-smi

In [ ]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

## 1. Клонирование репозитория

In [ ]:
%cd /content

# Если папка осталась от прошлой сессии — удаляем, чтобы клон был чистым.
!rm -rf rsna-pneumonia-yolo-pipeline

!git clone -b development https://github.com/e-eki/rsna-pneumonia-yolo-pipeline.git

In [ ]:
%cd rsna-pneumonia-yolo-pipeline
!git branch --show-current

## 2. Установка зависимостей

In [ ]:
!pip install -q -r requirements.txt

## 3. Настройка Kaggle API через Colab Secrets

Токен хранится в Colab Secrets (🔑 на левой панели), **не в коде**.

- Имя секрета: `KAGGLE_API_TOKEN`
- Значение: `KGAT_...`

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
    print("✅ KAGGLE_API_TOKEN загружен из Colab Secrets")
except Exception as e:
    raise RuntimeError(
        "Не удалось получить KAGGLE_API_TOKEN. "
        "Добавьте секрет с этим именем в 🔑 на левой панели Colab."
    ) from e

# Проверка, что CLI видит токен
!kaggle datasets metadata poeticmage/rsna-pneumonia-detection-challenge-train-val

## 4. Скачивание датасета

In [ ]:
!python -m src.download_data --config configs/config.yaml

In [ ]:
# Быстрая проверка структуры датасета
!echo "train images: $(ls data/raw/rsna_yolo/images/train | wc -l)"
!echo "train labels: $(ls data/raw/rsna_yolo/labels/train | wc -l)"
!echo "train labels с боксами: $(find data/raw/rsna_yolo/labels/train -name '*.txt' -size +0c | wc -l)"
!echo "val images:   $(ls data/raw/rsna_yolo/images/val | wc -l)"
!echo "val labels с боксами: $(find data/raw/rsna_yolo/labels/val -name '*.txt' -size +0c | wc -l)"

## 5. Подготовка датасета

Фильтрация «плохой» разметки, конвертация изображений, сборка `data/processed/` + `data.yaml`.

In [ ]:
# Полная пересборка — сносим прошлые артефакты, чтобы результат был детерминированным.
!rm -rf data/processed data/staged data/samples
!python -m src.prepare_yolo_dataset --config configs/config.yaml

In [ ]:
!ls data/processed
!ls data/processed/images/train | head -2
!ls data/processed/labels/train | head -2

In [ ]:
import pandas as pd
df = pd.read_csv("data/samples/rejected/rejected_annotations.csv")
print(df.to_string())

## 6. Статистика и визуальные проверки

In [ ]:
!python -m src.analyze_annotations --stage processed --split train
!python -m src.analyze_annotations --stage processed --split val

In [ ]:
!python -m src.sample_and_draw --stage processed --split train --num 50 --grid
!python -m src.sample_and_draw --stage processed --split val   --num 50 --grid

In [ ]:
!python -m src.collect_rejected --config configs/config.yaml

### 6.1. Просмотр сэмплов (слайдер)

Поменяйте `FOLDER`, чтобы посмотреть `train`, `val` или `rejected/drawn`.

In [ ]:
from pathlib import Path
from IPython.display import display, Image
import ipywidgets as widgets

FOLDER = Path("data/samples/processed/val")  # ← обработанные: processed/train или processed/val
paths = sorted(FOLDER.glob("*.jpg"))

if not paths:
    raise RuntimeError(f"Нет изображений в {FOLDER}")

slider = widgets.IntSlider(
    min=0, max=len(paths) - 1, step=1,
    description=f"0/{len(paths)-1}",
)
out = widgets.Output()

def render(idx):
    with out:
        out.clear_output(wait=True)
        print(paths[idx].name)
        display(Image(str(paths[idx]), width=800))

widgets.interactive_output(render, {"idx": slider})
display(slider, out)

## 7. Обучение

Параметры (модель, эпохи, batch, аугментации) — в `configs/config.yaml`.

`--run-name` пишите осмысленно: так проще сопоставлять `runs/train/<run>` и `runs/val/<run>`.

In [ ]:
!rm -rf runs/train runs/val
!python -m src.train --config configs/config.yaml --run-name smoke_yolov8s_30ep_no_mosaic

## 8. Оценка

Имя папки в `runs/val/` выводится автоматически из имени train-папки.

In [ ]:
!python -m src.evaluate --config configs/config.yaml

### 8.1. Кривые обучения

Слайдер по PNG-файлам в `runs/train/<run_name>/` (results, confusion_matrix, PR-кривые и т.п.).

In [ ]:
from pathlib import Path
from IPython.display import display, Image
import ipywidgets as widgets

train_runs = sorted(Path("runs/train").glob("*/"))
if not train_runs:
    raise RuntimeError("Нет прогонов в runs/train/")
FOLDER = train_runs[-1]  # последний прогон
paths = sorted(FOLDER.glob("*.png"))

if not paths:
    raise RuntimeError(f"Нет PNG в {FOLDER}")

slider = widgets.IntSlider(
    min=0, max=len(paths) - 1, step=1,
    description=f"0/{len(paths)-1}",
)
out = widgets.Output()

def render(idx):
    with out:
        out.clear_output(wait=True)
        print(paths[idx].name)
        display(Image(str(paths[idx]), width=800))

widgets.interactive_output(render, {"idx": slider})
display(slider, out)

## 9. Сохранение результатов на Google Drive

Упаковываем `runs/` и `data/reports/` — чтобы после отключения Colab-рантайма остались метрики и логи.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!tar -czf /content/drive/MyDrive/rsna-runs.tar.gz \
     -C /content/rsna-pneumonia-yolo-pipeline runs data/reports data/staged

!ls -lh /content/drive/MyDrive/rsna-runs.tar.gz

## 10. Итоговая сводка по пайплайну

In [ ]:
import json
from pathlib import Path
import pandas as pd

def fmt(v):
    try:
        return f"{float(v):.4f}"
    except (TypeError, ValueError):
        return str(v)

print("=" * 60)
print("СВОДКА ПО ПАЙПЛАЙНУ")
print("=" * 60)

# 1. Отчёты по processed
for split in ("train", "val"):
    report_path = Path("data/reports") / f"processed_{split}" / "report.json"
    if report_path.exists():
        data = json.load(open(report_path))
        bpi = data.get("boxes_per_image", {})
        print(f"\n[processed/{split}]")
        print(f"  боксов:                 {data.get('num_boxes')}")
        print(f"  изображений с боксами:  {bpi.get('with_boxes')}")
        print(f"  изображений без боксов: {bpi.get('without_boxes')}")
        if "yolo_out_of_bounds" in data:
            print(f"  YOLO out-of-bounds:     {data['yolo_out_of_bounds']}")

# 2. Отчёт фильтрации
filt = Path("data/staged/filtering_report.json")
if filt.exists():
    data = json.load(open(filt))
    print(f"\n[Фильтрация]")
    for s in data.get("splits", []):
        print(f"  [{s['split']}]")
        print(f"    изображений всего:   {s['images_total']}")
        print(f"    positive:            {s['images_positive']}")
        print(f"    negative:            {s['images_negative']}")
        print(f"    dropped_to_rejected: {s['images_dropped_to_rejected']}")
        print(f"    боксов сохранено:    {s['boxes_kept']}")
        print(f"    боксов отвергнуто:   {s['boxes_rejected']}")
        for reason, count in s.get("rejected_by_reason", {}).items():
            if count:
                print(f"      {reason}: {count}")

# 3. Метрики последнего train-прогона
train_runs = sorted(Path("runs/train").glob("*/results.csv"))
if train_runs:
    last_run = train_runs[-1]
    df = pd.read_csv(last_run)
    last = df.iloc[-1]
    print(f"\n[Обучение — {last_run.parent.name}]")
    print(f"  эпох:         {int(last.get('epoch', -1))}")
    print(f"  mAP@0.5:      {fmt(last.get('metrics/mAP50(B)'))}")
    print(f"  mAP@0.5:0.95: {fmt(last.get('metrics/mAP50-95(B)'))}")
    print(f"  precision:    {fmt(last.get('metrics/precision(B)'))}")
    print(f"  recall:       {fmt(last.get('metrics/recall(B)'))}")

# 4. Прогоны оценки
val_runs = sorted(Path("runs/val").glob("*/"))
if val_runs:
    print(f"\n[Оценка]")
    for run in val_runs[-3:]:
        print(f"  {run.name}")

print("\n" + "=" * 60)